# Hybrid MPC on a track with obstacles — a guided recipe

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/notebooks/demo_mpc_hybrid_track_lap.ipynb)

This notebook mirrors [`demo_mpc_hybrid_track_lap.py`](../scripts/hybrid/demo_mpc_hybrid_track_lap.py) as a **short textbook chapter**: we assemble spatial MPC ingredients, then close the loop with minilink's **hybrid computer framework** instead of a hand-written outer loop.

**The scenario:** a rate-input bicycle (`u` / `y` ports) laps a rounded rectangle, avoids sphere obstacles, and stays in the corridor — using **soft spatial costs** and **warm-start MPC** inside a `HybridDiagram`.

---

## The recipe at a glance

| Step | Ingredient (math) | Software helper |
| --- | --- | --- |
| 1 | Workspace: centerline, corridor tube, obstacle union SDF | `ReferenceTrack`, `Scene` |
| 2 | Planner plant $\dot{\mathbf{x}}=\mathbf{f}(\mathbf{x},\mathbf{u})$ and bounds | `JaxDynamicBicycleRateInputsUY` (`sys_mpc`) |
| 3 | Workspace cost preview (optional) | `as_cost` + heatmaps |
| 4 | Body probes $\mathbf{p}_k(\mathbf{x})$ | `bind(sys, car_outline)` |
| 5 | State fields $\phi(\mathbf{x})$ | `distance_field`, `corridor_field`, `clearance_field` |
| 6 | Soft running cost $g$ via shaping | `field.as_cost(shaping=...)` + `QuadraticCost` |
| 7 | Continuous planning slice | `PlanningProblem` |
| 8 | Compile-once MPC NLP | `TrajectoryOptimizationPlanner` (`transcription="direct_collocation"`) |
| 9 | Warm-start MPC block (packed $\mathbf{z}$ on `Computer.x`) | `ModelPredictiveController` |
| 10 | Schedule the computer | `computer = mpc @ sys_sim  # product hybrid` |
| 11 | Close the loop on the plant | `hybrid = computer @ sys_sim` |
| 12 | Co-simulate discrete MPC + continuous plant | `hybrid.compute_trajectory` |
| 13 | Trajectory plots + animation overlays | `plot_trajectory`, `mpc_animation_overlays` |

---

**Further reading** — spatial MPC pipeline: [`demo_mpc_spatial_scene_guide.ipynb`](demo_mpc_spatial_scene_guide.ipynb) (manual MPC loop). Here we replace Step 11 of that notebook with the **hybrid shortcuts** above.

Run from the minilink repository root with `pip install -e ".[jax]"`.


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from minilink.core.backends import configure_jax
from minilink.core.costs import QuadraticCost
from minilink.core.geometry import Sphere
from minilink.dynamics.catalog.vehicles.dynamic_bicycle import (
    JaxDynamicBicycleRateInputsUY,
)
from minilink.control.mpc import (
    ModelPredictiveController,
    mpc_animation_overlays,
)
from minilink.planning.problems import PlanningProblem
from minilink.planning.trajectory_optimization.planner import (
    TrajectoryOptimizationPlanner,
)
from minilink.planning.spatial.collision import bind, car_outline, point_probe
from minilink.planning.spatial.grid import pad_bounds, sample_field_costs
from minilink.planning.spatial.paths import from_waypoints
from minilink.planning.spatial.plotting import plot_cost_field_exports
from minilink.planning.spatial.scene import Scene
from minilink.planning.spatial.shaping import (
    inverse_barrier,
    quadratic_excess,
    quadratic_hinge,
)
from minilink.planning.spatial.track import ReferenceTrack
configure_jax(enable_x64=True)


## Mise en place — scenario parameters

Tuning constants for the compact loop (24 m × 14 m), three sphere keepouts, and MPC/hybrid rates. Same numbers as the script companion.

In [ ]:
U_TARGET = 12.0
MPC_DT = 0.1
MPC_HORIZON = 2.0
MPC_STEPS = 10

TF_SIM = 20.0
SIM_DT = 0.005

OBSTACLE_RADIUS = 0.2
OBSTACLE_MARGIN = 0.2
OBSTACLE_CENTERS = ((4.0, -6.2), (-2.0, 7.0), (12.0, 1.5))
TRACK_WIDTH = 24.0
TRACK_HEIGHT = 14.0
TURN_RADIUS = 3.5
CORRIDOR_HALF_WIDTH = 2.0

PATH_COST_WEIGHT = 10.0
CORRIDOR_COST_WEIGHT = 50.0
OBSTACLE_REPULSION_WEIGHT = 6.0
OBSTACLE_REPULSION_EPS = 0.08

START_XY = np.array([-(TRACK_WIDTH / 2 - TURN_RADIUS), -(TRACK_HEIGHT / 2)])
START_THETA = 0.001
VX0 = 2.5
PLOT_MARGIN = 3.0
PLOT_BOUNDS = (
    (-TRACK_WIDTH / 2 - PLOT_MARGIN, TRACK_WIDTH / 2 + PLOT_MARGIN),
    (-TRACK_HEIGHT / 2 - PLOT_MARGIN, TRACK_HEIGHT / 2 + PLOT_MARGIN),
)


def _rounded_rect_loop(
    cx=0.0, cy=0.0, width=TRACK_WIDTH, height=TRACK_HEIGHT, radius=TURN_RADIUS
):
    w2, h2, r = width / 2.0, height / 2.0, radius
    hx = w2 - r

    def arc(ccx, ccy, a0):
        t = np.linspace(a0, a0 + np.pi / 2, 8)
        return np.column_stack([ccx + r * np.cos(t), ccy + r * np.sin(t)])

    pts = [
        [[cx - hx, cy - h2]],
        np.linspace([cx - hx, cy - h2], [cx + hx, cy - h2], 4)[1:],
        arc(cx + hx, cy - h2 + r, -np.pi / 2)[1:],
        np.linspace([cx + w2, cy - h2 + r], [cx + w2, cy + h2 - r], 3)[1:-1],
        arc(cx + hx, cy + h2 - r, 0.0)[1:],
        np.linspace([cx + hx, cy + h2], [cx - hx, cy + h2], 4)[1:-1],
        arc(cx - hx, cy + h2 - r, np.pi / 2)[1:],
        np.linspace([cx - w2, cy + h2 - r], [cx - w2, cy - h2 + r], 3)[1:-1],
        arc(cx - hx, cy - h2 + r, np.pi)[1:],
        [[cx - hx, cy - h2]],
    ]
    return np.vstack(pts)

## Step 1 — Workspace geometry

Let the reference centerline be a polyline $\mathcal{C}$ with arc-length $s$, corridor half-width $w$. For a workspace point $\mathbf{p}$:

$$
d(\mathbf{p}) = \min_s \|\mathbf{p} - \mathcal{C}(s)\|, \qquad
m(\mathbf{p}) = w - d(\mathbf{p}), \qquad
c(\mathbf{p}) = \min_j \mathrm{sdf}_{O_j}(\mathbf{p}).
$$

| Object | Role |
| --- | --- |
| `track` | centerline + corridor width |
| `scene` | sphere obstacles → clearance $c(\mathbf{p})$ |

In [ ]:
loop_xy = _rounded_rect_loop()
track = ReferenceTrack(from_waypoints(loop_xy), half_width=CORRIDOR_HALF_WIDTH)
keepout_radius = OBSTACLE_RADIUS + OBSTACLE_MARGIN
scene = Scene(
    obstacles=[Sphere(center, keepout_radius) for center in OBSTACLE_CENTERS]
)

print(f"lap length = {track.path.total_length:.2f} m")
_, ax = scene.plot(show=False, bounds=PLOT_BOUNDS, show_density=False, title="")
track.plot(show=False, ax=ax, bounds=PLOT_BOUNDS, title="")
ax.scatter([START_XY[0]], [START_XY[1]], c="C1", s=36, zorder=8, label="start")
ax.set_title("Workspace: track tube + keepouts")
ax.legend(loc="upper left", fontsize=8)
plt.show()

## Step 2 — Planner plant (`sys_mpc`)

`JaxDynamicBicycleRateInputsUY` is the **catalog bicycle with standard `u` / `y` ports** — no Demux wiring in the hybrid diagram. Wheel speed $\omega_r$ and steer $\delta$ are states; their rates are the control $\mathbf{u}$.

Bounds on `sys_mpc` define the NLP feasible set inside `TrajectoryOptimizationPlanner`. The simulation plant `sys_sim` (Step 11) can differ slightly (mass / inertia mismatch).

In [ ]:
sys_mpc = JaxDynamicBicycleRateInputsUY()
sys_mpc.state.lower_bound[6] = 0.0
sys_mpc.state.upper_bound[6] = 90.0
sys_mpc.state.lower_bound[7] = -0.55
sys_mpc.state.upper_bound[7] = 0.55
sys_mpc.inputs["u"].lower_bound = np.array([-80.0, -2.0])
sys_mpc.inputs["u"].upper_bound = np.array([80.0, 2.0])
print(f"planner plant: n={sys_mpc.n}, m={sys_mpc.m}")

## Step 3 — Workspace cost heatmaps (optional preview)

Rasterize $\ell(\mathbf{p}) = w\, s(\phi(\mathbf{p}))$ before attaching the car body. Useful for tuning weights before running hybrid MPC.

In [ ]:
probe = bind(sys_mpc, point_probe())
path_cost_viz = track.distance_field(probe).as_cost(
    weight=PATH_COST_WEIGHT, shaping=quadratic_excess(threshold=0.1)
)
corridor_cost_viz = track.corridor_field(probe).as_cost(
    weight=CORRIDOR_COST_WEIGHT, shaping=quadratic_hinge(threshold=0.0)
)
obstacle_cost_viz = scene.clearance_field(probe).as_cost(
    weight=OBSTACLE_REPULSION_WEIGHT,
    shaping=inverse_barrier(epsilon=OBSTACLE_REPULSION_EPS),
)

cost_bounds = pad_bounds(PLOT_BOUNDS, 1.0)
_cost_kw = dict(bounds=cost_bounds, state_dim=sys_mpc.n, grid=(100, 100))
layers = {
    "combined": sample_field_costs(
        [path_cost_viz, corridor_cost_viz, obstacle_cost_viz], **_cost_kw
    ),
}
plot_cost_field_exports(
    layers,
    track=track,
    scene=scene,
    overlay_bounds=PLOT_BOUNDS,
    log_scale=True,
)

## Steps 4–6 — Body, state fields, soft cost

Forward kinematics maps $\mathbf{x}$ to body sphere probes; state fields aggregate clearance / path / corridor over probes. The running cost is

$$
g = g_{\mathrm{quad}} + w_p s_{\mathrm{path}} + w_c s_{\mathrm{cor}} + w_o s_{\mathrm{obs}}.
$$

We use **soft** spatial terms (not hard `X` constraints) — same choice as the spatial scene guide on this short horizon.

In [ ]:
r_r = sys_mpc.params["r_r"]
x_cruise = np.array([0.0, 0.0, 0.0, U_TARGET, 0.0, 0.0, U_TARGET / r_r, 0.0])
body = bind(sys_mpc, car_outline(length=2.4, width=0.2, margin=0.05))

cost = (
    QuadraticCost.from_system(
        sys_mpc,
        Q=np.diag([0.0, 0.0, 0.0, 0.15, 4.0, 6.0, 0.1, 80.0]),
        R=np.diag([1.0, 22.0]),
        S=np.diag([0.0, 0.0, 0.0, 0.15, 4.0, 6.0, 0.1, 80.0]),
        xbar=x_cruise,
        ubar=np.zeros(2),
    )
    + track.distance_field(body).as_cost(
        weight=PATH_COST_WEIGHT, shaping=quadratic_excess(threshold=0.1)
    )
    + track.corridor_field(body).as_cost(
        weight=CORRIDOR_COST_WEIGHT, shaping=quadratic_hinge(threshold=0.0)
    )
    + scene.clearance_field(body).as_cost(
        weight=OBSTACLE_REPULSION_WEIGHT,
        shaping=inverse_barrier(epsilon=OBSTACLE_REPULSION_EPS),
    )
)

x0 = np.array(
    [START_XY[0], START_XY[1], START_THETA, VX0, 0.0, 0.0, VX0 / r_r, 0.0]
)
problem = PlanningProblem(sys=sys_mpc, x_start=x0, cost=cost, tf=MPC_HORIZON)
print(problem)


## Step 8 — Compile-once `TrajectoryOptimizationPlanner`

At each MPC tick we solve a finite-horizon NLP in decision vector $\mathbf{z}$, pinned at the measured state $\mathbf{y}$. `TrajectoryOptimizationPlanner` **JIT-compiles** the parametric program once; `solve_trajectory_from(y) / compute_command` re-solves with warm-start.

| Object | Role |
| --- | --- |
| `transcription="direct_collocation"` | method seam (string preset) |
| flat planner kwargs | `n_steps`, JAX backend + SLSQP knobs |
| `TrajectoryOptimizationPlanner` | `compile_parametric_program()` once, `solve_trajectory_from(y)` each fire |


In [ ]:
planner = TrajectoryOptimizationPlanner(
    problem,
    n_steps=MPC_STEPS,
    transcription="direct_collocation",
    compile_backend="jax",
    record_solve_time=True,
    optimizer_method="scipy_slsqp",
    optimizer_options={"maxiter": 120, "ftol": 1.0},
)
print("planner ready (parametric compile on ModelPredictiveController construction)")


## Step 9 — Warm-start block: `ModelPredictiveController`

**Stateful vs stateless.**

| Block | `Computer.x` | Warm-start |
| --- | --- | --- |
| `ModelPredictiveController(..., warm_start=False)` | empty (algebraic) | default guess each tick |
| `ModelPredictiveController` | packed $\mathbf{z}$ | shift previous plan, pin $\mathbf{x}_0=\mathbf{y}$ |

The stateful block is a `StepSystem`: `step` commits the latched $\mathbf{z}$; port outputs `u_ff`, `x_ff`, `z` share one NLP per tick.

In [ ]:
mpc = ModelPredictiveController(planner, dt_mpc=MPC_DT, warm_start=True, step_disp=True)
print(f"MPC block state dim = {mpc.n} (packed z)")

## Step 10 — Simulation plant (`sys_sim`)

Hybrid co-simulation integrates the **continuous** plant between MPC ticks. A slight mass/inertia mismatch vs `sys_mpc` mimics model error.

In [ ]:
sys_sim = JaxDynamicBicycleRateInputsUY()
sys_sim.params["mass"] = 1.03 * sys_mpc.params["mass"]
sys_sim.params["inertia"] = 1.02 * sys_mpc.params["inertia"]
sys_sim.camera_scale = 16.0
sys_sim.x0 = x0.copy()

## Steps 11–12 — Hybrid composition

Two-layer API (same as the script):

```python
hybrid = mpc @ sys_sim   # schedule via mpc.dt_mpc; auto-wire y → MPC, u_ff → plant
```

| Layer | Math picture | minilink |
| --- | --- | --- |
| Computer | MPC fires every $\Delta t_{\mathrm{MPC}}$; holds $\mathbf{z}_k$ | `Computer` + `StepSchedule` |
| Plant | $\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x}, \mathbf{u}_{\mathrm{ff}})$ between fires | `sys_sim` in `HybridDiagram` |
| Boundary | $\mathbf{y}_k = \mathbf{h}(\mathbf{x}(t_k))$ | `plant_to_computer` / `computer_to_plant` |


In [ ]:
hybrid = mpc @ sys_sim
hybrid.plot_diagram()


## Step 13 — `compute_trajectory`

`HybridSimulator` runs the coupled rollout: fine plant steps at `SIM_DT`, MPC ticks at `MPC_DT`. Omit `x0_computer` — warm-start MPC already packs default $\mathbf{z}$ on `Computer.x` / diagram `x0`.


In [ ]:
result = hybrid.compute_trajectory(
    tf=TF_SIM,
    x0_plant=x0,
    plant_dt_inner=SIM_DT,
    compile_backend="jax",
)

traj = result.plant
clearances = [
    np.hypot(traj.x[0, :] - cx, traj.x[1, :] - cy) - OBSTACLE_RADIUS
    for cx, cy in OBSTACLE_CENTERS
]
print(
    f"done: mean vx={float(np.mean(traj.x[3, :])):.2f} m/s, "
    f"min obstacle clearance={float(np.min(clearances)):.2f} m"
)

## Step 14 — Results

Executed path on the scene, state/input histories, and animation with track corridor + obstacle skin + MPC horizon overlay (`mpc_animation_overlays`).

In [ ]:
fig, ax = track.plot(
    show=False, bounds=PLOT_BOUNDS, title="Hybrid MPC executed path"
)
scene.plot(show=False, ax=ax, bounds=PLOT_BOUNDS, show_density=False, title=None)
ax.plot(traj.x[0, :], traj.x[1, :], color="tab:blue", linewidth=1.8, label="executed")
ax.scatter([START_XY[0]], [START_XY[1]], c="C1", s=36, zorder=8, label="start")
ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()
plt.show()

hybrid.plot_trajectory()


In [ ]:
hybrid.animate(
    overlays=mpc_animation_overlays(result, planner, scene=scene, track=track)
)

---

## Summary

**Pipeline:** workspace geometry → state-field costs → `PlanningProblem` → `TrajectoryOptimizationPlanner` → `ModelPredictiveController` → `mpc @ sys_sim  # product hybrid` → `computer @ sys_sim` → `compute_trajectory` → overlays.

Script twin: [`demo_mpc_hybrid_track_lap.py`](../scripts/hybrid/demo_mpc_hybrid_track_lap.py). Spatial MPC theory (manual loop): [`demo_mpc_spatial_scene_guide.ipynb`](demo_mpc_spatial_scene_guide.ipynb).